# RQ2/RQ3/RQ4 - Meta-feature Selection, Efficiency, and Ablation

This notebook reconstructs the article tables and runtime figure from the released CSV files.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS = ROOT / 'results'
FIGURES = ROOT / 'figures'
REPRODUCED = FIGURES / 'reproduced'
REPRODUCED.mkdir(exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 300,
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

def save_reproduced(fig, name):
    fig.savefig(REPRODUCED / f'{name}.png', bbox_inches='tight')
    fig.savefig(REPRODUCED / f'{name}.pdf', bbox_inches='tight')

def pm(mean, std, digits=2):
    return f'{mean:.{digits}f} ± {std:.{digits}f}'

def show_article_figure(filename):
    display(Markdown(f'**Article figure:** `figures/{filename}`'))
    display(Image(filename=str(FIGURES / filename)))

RQ2 = RESULTS / 'rq2_meta_feature_selection'
RQ3 = RESULTS / 'rq3_efficiency'
RQ4 = RESULTS / 'rq4_family_ablation'


## Selected Reduced Meta-features

In [ ]:
selected = pd.read_csv(RQ2 / 'selected_features_pearson085_random_forest_importance_k10.csv')
display(selected)
print(selected.to_latex(index=False, escape=False))

## Table 4 - Full versus Reduced MetaMatch

In [ ]:
perf = pd.read_csv(RQ2 / 'rq2_rq3_table_for_paper.csv')
rows = perf[perf['config'].isin(['full60', 'reduced10'])].copy()
rows['config'] = pd.Categorical(rows['config'], ['full60', 'reduced10'], ordered=True)
rows = rows.sort_values('config')
eff = pd.read_csv(RQ3 / 'table_efficiency_preliminary_application_total_551_pairs_with_reduced.csv')
lookup = eff.set_index('method')

paper_selection_table = pd.DataFrame({
    'Set': ['Full', 'Reduced'],
    '# meta-features': rows['n_features'].astype(int).tolist(),
    'F1': [pm(m, s) for m, s in zip(rows['mean_f1'], rows['std_f1'])],
    'Precision': [pm(m, s) for m, s in zip(rows['mean_precision'], rows['std_precision'])],
    'Recall': [pm(m, s) for m, s in zip(rows['mean_recall'], rows['std_recall'])],
    'Generation time (s)': [66251.24, 5424.98],
    'Random Forest train+test (s)': [249.47, 195.28],
    'Total time (s)': [66500.71, 5620.26],
})
display(paper_selection_table)
print(paper_selection_table.to_latex(index=False, escape=False))

## Table 5 - Efficiency Summary

In [ ]:
efficiency = pd.read_csv(RQ3 / 'table_efficiency_preliminary_application_total_551_pairs_with_reduced.csv')
method_order = ['Full MetaMatch', 'Reduced MetaMatch', 'SMUTF', 'LLMATCH', 'MagnetoFTGPT', 'MagnetoFT', 'MagnetoGPT', 'Magneto', 'ISResMat', 'COMA++', 'COMA', 'Similarity Flooding', 'Distribution Based', 'Cupid']
efficiency['method'] = pd.Categorical(efficiency['method'], method_order, ordered=True)
efficiency = efficiency.sort_values('method')
paper_efficiency_table = pd.DataFrame({
    'Method': efficiency['method'].astype(str),
    'Preliminary time (s)': efficiency['preliminary_time_sec'].fillna(0).map(lambda x: f'{x:.2f}'),
    'Application time (s)': efficiency['application_time_sec_551_pairs'].fillna(0).map(lambda x: f'{x:.2f}'),
    'Total time (s)': efficiency['known_total_time_sec_551_pairs'].fillna(0).map(lambda x: f'{x:.2f}'),
})
display(paper_efficiency_table)
print(paper_efficiency_table.to_latex(index=False, escape=False))

## Figure 4 - Total Runtime

In [ ]:
plot_df = efficiency.sort_values('known_total_time_sec_551_pairs', ascending=True).copy()
colors = ['#1F77B4' if 'MetaMatch' in str(m) else '#A6A6A6' for m in plot_df['method']]
fig, ax = plt.subplots(figsize=(8, 5.5))
bars = ax.barh(plot_df['method'].astype(str), plot_df['known_total_time_hours_551_pairs'], color=colors)
ax.set_xlabel('Total runtime (hours)')
ax.grid(axis='x', alpha=0.25)
for bar, value in zip(bars, plot_df['known_total_time_hours_551_pairs']):
    ax.text(value + max(plot_df['known_total_time_hours_551_pairs']) * 0.01, bar.get_y() + bar.get_height()/2, f'{value:.2f}', va='center', fontsize=8)
fig.tight_layout()
save_reproduced(fig, 'fig_rq4_total_runtime_bar')
plt.show()
show_article_figure('fig_rq4_total_runtime_bar.png')

## Table 6 - Family Ablation

In [ ]:
ablation = pd.read_csv(RQ4 / 'family_ablation_complete_vs_reduced_for_paper.csv')
paper_ablation_table = pd.DataFrame({
    'Meta-feature set': ablation['feature_set'],
    'Full F1': [pm(m, s) for m, s in zip(ablation['complete_f1_mean'], ablation['complete_f1_std'])],
    'Full Precision': [pm(m, s) for m, s in zip(ablation['complete_precision_mean'], ablation['complete_precision_std'])],
    'Full Recall': [pm(m, s) for m, s in zip(ablation['complete_recall_mean'], ablation['complete_recall_std'])],
    'Reduced F1': [pm(m, s) for m, s in zip(ablation['reduced_f1_mean'], ablation['reduced_f1_std'])],
    'Reduced Precision': [pm(m, s) for m, s in zip(ablation['reduced_precision_mean'], ablation['reduced_precision_std'])],
    'Reduced Recall': [pm(m, s) for m, s in zip(ablation['reduced_recall_mean'], ablation['reduced_recall_std'])],
})
display(paper_ablation_table)
print(paper_ablation_table.to_latex(index=False, escape=False))